In [1]:
"""
Evaluate the saved LightGBM model (lgbm_t2m_sst.txt) against unseen 2024-2025 data.

This mirrors the feature-engineering pipeline in model_training_final.ipynb
(cells 3-4) exactly, then runs inference instead of training, and reports
error metrics overall and split by land vs. sea.

USAGE:
    Point `data_dir` at the folder holding your 2024-2025 .nc files
    (these should NOT be the same files used for training/val/test).
    Point `model_path` at lgbm_t2m_sst.txt.
"""

'\nEvaluate the saved LightGBM model (lgbm_t2m_sst.txt) against unseen 2024-2025 data.\n\nThis mirrors the feature-engineering pipeline in model_training_final.ipynb\n(cells 3-4) exactly, then runs inference instead of training, and reports\nerror metrics overall and split by land vs. sea.\n\nUSAGE:\n    Point `data_dir` at the folder holding your 2024-2025 .nc files\n    (these should NOT be the same files used for training/val/test).\n    Point `model_path` at lgbm_t2m_sst.txt.\n'

In [2]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import xarray as xr
import lightgbm as lgb



In [3]:
repo_root = Path(r"C:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026")
data_dir = repo_root / "data" / "processed"        # <-- adjust if different
model_path = repo_root / "models" / "lgbm_t2m_sst.txt"

target_years = [2024, 2025]
keep_row = 1.0       # fraction of valid target-year rows to keep (1.0 = all)
land_threshold = 0.5
seed = 42

vars = ["ssrd", "strd", "tcc", "lsm", "z",
        "lat", "lon", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]



In [4]:
size = model_path.stat().st_size
print(f"Size on disk: {size:,} bytes")

with open(model_path, "rb") as f:
    head = f.read(300)
print(head)

Size on disk: 36,061,960 bytes
b'tree\nversion=v4\nnum_class=1\nnum_tree_per_iteration=1\nlabel_index=0\nmax_feature_idx=10\nobjective=regression\nfeature_names=Column_0 Column_1 Column_2 Column_3 Column_4 Column_5 Column_6 Column_7 Column_8 Column_9 Column_10\nfeature_infos=[-1.9015655517578125:4140416] [282592.0625:1870789.5] [0:1] [0:1]'


In [ ]:
if not model_path.exists():
    print(f"models folder exists: {model_path.parent.exists()}")
    if model_path.parent.exists():
        print("Files actually in that folder:")
        for p in model_path.parent.iterdir():
            print(" ", repr(p.name))
    raise FileNotFoundError(f"No file found at {model_path}")

booster = lgb.Booster(model_file=str(model_path))
print(f"Loaded model from {model_path} "
      f"({booster.num_trees()} trees, {booster.num_feature()} features)")

Loaded model from C:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\models\lgbm_t2m_sst.txt (3000 trees, 11 features)


In [ ]:
files = sorted(glob.glob(str(data_dir / "*.nc")))
if not files:
    raise FileNotFoundError(f"No .nc files under {data_dir}")
print(f"Found {len(files)} files total (mixed years)")

with xr.open_dataset(files[0], engine="netcdf4") as first:
    lat = first["latitude"].values.astype(np.float32)
    lon = first["longitude"].values.astype(np.float32)
    lsm2d = first["lsm"].isel(valid_time=0).values.astype(np.float32)
    z2d = first["z"].isel(valid_time=0).values.astype(np.float32)

n_cell = lat.size * lon.size
lon2d, lat2d = np.meshgrid(lon, lat)
lat_flat = lat2d.ravel()
lon_flat = lon2d.ravel()
lsm_flat = lsm2d.ravel()
z_flat = z2d.ravel()
is_land = lsm_flat >= land_threshold
print(f"Grid: {lat.size} x {lon.size} = {n_cell:,} cells "
      f"({int(is_land.sum()):,} land)")



Found 120 files total (mixed years)


Grid: 125 x 121 = 15,125 cells (8,385 land)


In [ ]:
'''
rng = np.random.default_rng(seed)
x_parts, y_parts, land_parts, yr_parts = [], [], [], []
files_used = 0
years_seen = set()

for i, f in enumerate(files, 1):
    with xr.open_dataset(f, engine="netcdf4") as blk_full:
        # Cheap: valid_time is a coordinate, this doesn't load t2m/ssrd/etc.
        all_times = pd.DatetimeIndex(blk_full["valid_time"].values)
        all_years = all_times.year.to_numpy()
        years_seen.update(np.unique(all_years).tolist())
        mask = np.isin(all_years, target_years)

        if not mask.any():
            continue  # skip this file entirely -- no target-year rows

        files_used += 1
        idx = np.where(mask)[0]
        blk = blk_full.isel(valid_time=idx)  # only load target rows below
        nb = blk.sizes["valid_time"]

        t2m = blk["t2m"].values.astype(np.float32).reshape(nb, -1)
        sst = blk["sst"].values.astype(np.float32).reshape(nb, -1)
        ssrd = blk["ssrd"].values.astype(np.float32).reshape(nb, -1)
        strd = blk["strd"].values.astype(np.float32).reshape(nb, -1)
        tcc = blk["tcc"].values.astype(np.float32).reshape(nb, -1)
        times = pd.DatetimeIndex(blk["valid_time"].values)

    target = np.where(is_land[None, :], t2m, sst)

    hour = times.hour.to_numpy(dtype=np.float32)
    doy = times.dayofyear.to_numpy(dtype=np.float32)
    year = times.year.to_numpy(dtype=np.int16)

    h_sin = np.sin(2 * np.pi * hour / 24).astype(np.float32)
    h_cos = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    d_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    d_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    def tile_t(a):
        return np.repeat(a[:, None], n_cell, axis=1)

    def tile_c(a):
        return np.repeat(a[None, :], nb, axis=0)

    xb = np.stack([ssrd, strd, tcc,
                   tile_c(lsm_flat), tile_c(z_flat),
                   tile_c(lat_flat), tile_c(lon_flat),
                   tile_t(h_sin), tile_t(h_cos),
                   tile_t(d_sin), tile_t(d_cos)], axis=-1)

    xb = xb.reshape(-1, len(vars))
    yb = target.ravel()
    yrb = np.repeat(year, n_cell)
    landb = np.tile(is_land, nb)

    keep = ~np.isnan(yb)
    if keep_row < 1.0:
        keep &= rng.random(yb.size) < keep_row
    x_parts.append(xb[keep])
    y_parts.append(yb[keep])
    land_parts.append(landb[keep])
    yr_parts.append(yrb[keep])

    print(f"  [{i}/{len(files)}] {Path(f).name}: "
          f"{nb} target-year timesteps kept "
          f"({sorted(set(year.tolist()))})")

print(f"\nYears found across all {len(files)} files: {sorted(years_seen)}")
print(f"Files containing target years {target_years}: {files_used}")

if not x_parts:
    raise RuntimeError(
        f"No rows found for target years {target_years}. "
        f"Years actually present in the data: {sorted(years_seen)}"
    )

x_test = np.concatenate(x_parts)
y_test = np.concatenate(y_parts)
land_test = np.concatenate(land_parts)
yr_test = np.concatenate(yr_parts)
del x_parts, y_parts, land_parts, yr_parts
print(f"Eval table: {x_test.shape[0]:,} rows x {x_test.shape[1]} features")
'''

'\nrng = np.random.default_rng(seed)\nx_parts, y_parts, land_parts, yr_parts = [], [], [], []\nfiles_used = 0\nyears_seen = set()\n\nfor i, f in enumerate(files, 1):\n    with xr.open_dataset(f, engine="netcdf4") as blk_full:\n        # Cheap: valid_time is a coordinate, this doesn\'t load t2m/ssrd/etc.\n        all_times = pd.DatetimeIndex(blk_full["valid_time"].values)\n        all_years = all_times.year.to_numpy()\n        years_seen.update(np.unique(all_years).tolist())\n        mask = np.isin(all_years, target_years)\n\n        if not mask.any():\n            continue  # skip this file entirely -- no target-year rows\n\n        files_used += 1\n        idx = np.where(mask)[0]\n        blk = blk_full.isel(valid_time=idx)  # only load target rows below\n        nb = blk.sizes["valid_time"]\n\n        t2m = blk["t2m"].values.astype(np.float32).reshape(nb, -1)\n        sst = blk["sst"].values.astype(np.float32).reshape(nb, -1)\n        ssrd = blk["ssrd"].values.astype(np.float32).resh

In [8]:
import time
import gc

rng = np.random.default_rng(seed)

y_true_list = []
y_pred_list = []
land_list = []
yr_list = []

print("Running streamed inference file-by-file...\n")

for i, f in enumerate(files, 1):
    t_start = time.time()
    
    # 1. Quick check if file contains target years without full load
    ds_meta = xr.open_dataset(f, engine="h5netcdf", decode_times=True)
    all_years = pd.DatetimeIndex(ds_meta["valid_time"].values).year.to_numpy()
    mask = np.isin(all_years, target_years)
    ds_meta.close()

    if not mask.any():
        continue

    # 2. Read only target slice
    idx = np.where(mask)[0]
    with xr.open_dataset(f, engine="h5netcdf") as blk_full:
        blk = blk_full.isel(valid_time=idx).load()  # force immediate load & release file lock
        
    nb = blk.sizes["valid_time"]

    t2m = blk["t2m"].values.astype(np.float32).reshape(nb, -1)
    sst = blk["sst"].values.astype(np.float32).reshape(nb, -1)
    ssrd = blk["ssrd"].values.astype(np.float32).reshape(nb, -1)
    strd = blk["strd"].values.astype(np.float32).reshape(nb, -1)
    tcc = blk["tcc"].values.astype(np.float32).reshape(nb, -1)
    times = pd.DatetimeIndex(blk["valid_time"].values)
    blk.close()

    target = np.where(is_land[None, :], t2m, sst)

    hour = times.hour.to_numpy(dtype=np.float32)
    doy = times.dayofyear.to_numpy(dtype=np.float32)
    year = times.year.to_numpy(dtype=np.int16)

    h_sin = np.sin(2 * np.pi * hour / 24).astype(np.float32)
    h_cos = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    d_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    d_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    def tile_t(a): return np.repeat(a[:, None], n_cell, axis=1)
    def tile_c(a): return np.repeat(a[None, :], nb, axis=0)

    xb = np.stack([ssrd, strd, tcc,
                   tile_c(lsm_flat), tile_c(z_flat),
                   tile_c(lat_flat), tile_c(lon_flat),
                   tile_t(h_sin), tile_t(h_cos),
                   tile_t(d_sin), tile_t(d_cos)], axis=-1).reshape(-1, len(vars))

    yb = target.ravel()
    yrb = np.repeat(year, n_cell)
    landb = np.tile(is_land, nb)

    keep = ~np.isnan(yb)
    if keep_row < 1.0:
        keep &= rng.random(yb.size) < keep_row

    xb = xb[keep]
    yb = yb[keep]
    landb = landb[keep]
    yrb = yrb[keep]

    t_read = time.time() - t_start

    # 3. Predict on single file
    t_pred_start = time.time()
    y_hat_b = booster.predict(xb)
    t_pred = time.time() - t_pred_start

    y_true_list.append(yb)
    y_pred_list.append(y_hat_b)
    land_list.append(landb)
    yr_list.append(yrb)

    print(f"[{i:3d}/{len(files)}] {Path(f).name[:15]}... | Rows: {yb.size:,} | Read: {t_read:.1f}s | Predict: {t_pred:.1f}s")

    del xb, yb, landb, yrb, y_hat_b, t2m, sst, ssrd, strd, tcc, blk
    gc.collect()

# ----------------------------------------------------------------------
# Concatenate and Report
# ----------------------------------------------------------------------
y_test = np.concatenate(y_true_list)
y_hat = np.concatenate(y_pred_list)
land_test = np.concatenate(land_list)
yr_test = np.concatenate(yr_list)

del y_true_list, y_pred_list, land_list, yr_list
gc.collect()

def report(name, y_true, y_pred):
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    print(f"{name:<28} n={len(y_true):>10,}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")

print("\n=== Overall ===")
report("All cells (t2m+sst)", y_test, y_hat)

print("\n=== By surface type ===")
report("Land only (t2m)", y_test[land_test], y_hat[land_test])
report("Sea only (sst)", y_test[~land_test], y_hat[~land_test])

print("\n=== By year ===")
for yr in np.unique(yr_test):
    m = yr_test == yr
    report(f"Year {yr}", y_test[m], y_hat[m])
    report(f"  Year {yr} - land", y_test[m & land_test], y_hat[m & land_test])
    report(f"  Year {yr} - sea", y_test[m & ~land_test], y_hat[m & ~land_test])

Running streamed inference file-by-file...

[  7/120] 217ec2c51f4e088... | Rows: 10,859,040 | Read: 2.7s | Predict: 181.7s
[  9/120] 23d106aefa17512... | Rows: 11,221,008 | Read: 2.5s | Predict: 201.8s
[ 27/120] 3ee403f76674f19... | Rows: 11,221,008 | Read: 2.4s | Predict: 212.1s
[ 30/120] 43625bd187db67f... | Rows: 11,221,008 | Read: 2.5s | Predict: 223.1s
[ 36/120] 53a48fe678058aa... | Rows: 11,221,008 | Read: 2.4s | Predict: 202.5s
[ 39/120] 5be819c56c99a31... | Rows: 11,221,008 | Read: 2.3s | Predict: 208.8s
[ 44/120] 69e643944361e7a... | Rows: 10,859,040 | Read: 2.4s | Predict: 202.0s
[ 48/120] 6e328a8bffa7ed0... | Rows: 11,221,008 | Read: 2.5s | Predict: 199.5s
[ 61/120] 8548449454ed6db... | Rows: 11,221,008 | Read: 2.5s | Predict: 197.9s
[ 70/120] 98a4e6bb6ea05f6... | Rows: 11,221,008 | Read: 2.5s | Predict: 203.3s
[ 75/120] a11d231aaf86c93... | Rows: 11,221,008 | Read: 2.5s | Predict: 200.7s
[ 77/120] a1f50ee9106fb9b... | Rows: 10,859,040 | Read: 2.5s | Predict: 192.8s
[ 78/120

In [ ]:
'''
def report(name, y_true, y_pred):
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    print(f"{name:<28} n={len(y_true):>10,}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")

n_samples = x_test.shape[0]
batch_size = 10_000_000  # Process 10M rows at a time to prevent RAM swapping

# Pre-allocate output array
y_hat = np.empty(n_samples, dtype=np.float32)

print(f"Starting batch prediction across {n_samples:,} rows...")

for start_idx in range(0, n_samples, batch_size):
    end_idx = min(start_idx + batch_size, n_samples)
    y_hat[start_idx:end_idx] = booster.predict(x_test[start_idx:end_idx])
    print(f" Predicted {end_idx:,} / {n_samples:,} rows...")

print("\n=== Overall ===")
report("All cells (t2m+sst)", y_test, y_hat)

print("\n=== By surface type ===")
report("Land only (t2m)", y_test[land_test], y_hat[land_test])
report("Sea only (sst)", y_test[~land_test], y_hat[~land_test])

print("\n=== By year ===")
for yr in np.unique(yr_test):
    m = yr_test == yr
    report(f"Year {yr}", y_test[m], y_hat[m])
    report(f"  Year {yr} - land", y_test[m & land_test], y_hat[m & land_test])
    report(f"  Year {yr} - sea", y_test[m & ~land_test], y_hat[m & ~land_test])
    '''

'\ndef report(name, y_true, y_pred):\n    err = y_true - y_pred\n    mae = np.mean(np.abs(err))\n    rmse = np.sqrt(np.mean(err ** 2))\n    ss_res = np.sum(err ** 2)\n    ss_tot = np.sum((y_true - y_true.mean()) ** 2)\n    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")\n    print(f"{name:<28} n={len(y_true):>10,}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")\n\nn_samples = x_test.shape[0]\nbatch_size = 10_000_000  # Process 10M rows at a time to prevent RAM swapping\n\n# Pre-allocate output array\ny_hat = np.empty(n_samples, dtype=np.float32)\n\nprint(f"Starting batch prediction across {n_samples:,} rows...")\n\nfor start_idx in range(0, n_samples, batch_size):\n    end_idx = min(start_idx + batch_size, n_samples)\n    y_hat[start_idx:end_idx] = booster.predict(x_test[start_idx:end_idx])\n    print(f" Predicted {end_idx:,} / {n_samples:,} rows...")\n\nprint("\n=== Overall ===")\nreport("All cells (t2m+sst)", y_test, y_hat)\n\nprint("\n=== By surface type ===")\nreport("Land 

In [ ]:
'''
def report(name, y_true, y_pred):
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    print(f"{name:<28} n={len(y_true):>10,}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")

y_hat = booster.predict(x_test)

print("\n=== Overall ===")
report("All cells (t2m+sst)", y_test, y_hat)

print("\n=== By surface type ===")
report("Land only (t2m)", y_test[land_test], y_hat[land_test])
report("Sea only (sst)", y_test[~land_test], y_hat[~land_test])

print("\n=== By year ===")
for yr in np.unique(yr_test):
    m = yr_test == yr
    report(f"Year {yr}", y_test[m], y_hat[m])
    report(f"  Year {yr} - land", y_test[m & land_test], y_hat[m & land_test])
    report(f"  Year {yr} - sea", y_test[m & ~land_test], y_hat[m & ~land_test])
'''

'\ndef report(name, y_true, y_pred):\n    err = y_true - y_pred\n    mae = np.mean(np.abs(err))\n    rmse = np.sqrt(np.mean(err ** 2))\n    ss_res = np.sum(err ** 2)\n    ss_tot = np.sum((y_true - y_true.mean()) ** 2)\n    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")\n    print(f"{name:<28} n={len(y_true):>10,}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")\n\ny_hat = booster.predict(x_test)\n\nprint("\n=== Overall ===")\nreport("All cells (t2m+sst)", y_test, y_hat)\n\nprint("\n=== By surface type ===")\nreport("Land only (t2m)", y_test[land_test], y_hat[land_test])\nreport("Sea only (sst)", y_test[~land_test], y_hat[~land_test])\n\nprint("\n=== By year ===")\nfor yr in np.unique(yr_test):\n    m = yr_test == yr\n    report(f"Year {yr}", y_test[m], y_hat[m])\n    report(f"  Year {yr} - land", y_test[m & land_test], y_hat[m & land_test])\n    report(f"  Year {yr} - sea", y_test[m & ~land_test], y_hat[m & ~land_test])\n'